# Phase 6 (post hoc): v5 checkpoints on the external databases (phase6_rr_baseline, external part)

PREREGISTRATION §10 v5, P5 with the P2 rules. The 36 v5 checkpoints chosen on validation and the two v2 references on
INCART and SVDB, with the v2 RR ratio computed from each database's annotations. No MIT-BIH test data is read, and nothing
is trained or chosen. CPU only.

1. Settings: **Internet on**. Accelerator: **None** (CPU). Add the dataset `ecgaug-v5-external` (a `data/` folder with
   `raw/incartdb`, `raw/svdb` and `processed/external`, and a `runs/` folder with the v5 checkpoints and reference models).
   Save Version > Save & Run All.
2. `RUN_TESTS = True` runs the full test suite at this commit first and stops if it fails.
3. Download the zip in Output and run `python kaggle/ingest_runs.py <zip>`.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
DATA_DATASET_DIR = "/kaggle/input/ecgaug-v5-external"   # attached dataset: data/ (external databases) and runs/ (v5 checkpoints)
DATABASE = "all"                      # "incartdb", "svdb" or "all"
RUN_TESTS = True                      # the full test suite first; nothing else runs if it fails
GIT_REF = "main"                      # branch name or full commit SHA to run
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
configs = ["configs/phase6_rr_baseline/rr_baseline.yaml"]
extra = ["--part", "external", "--database", DATABASE]
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase6_rr_baseline", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs, "--raw-dir", RAW_DATASET_DIR, "--artifacts-dir", DATA_DATASET_DIR, "--data-dir", DATA_DATASET_DIR]
if RUN_TESTS:
    cmd += ["--tests"]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO, env=env).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
